In [4]:
import pandas as pd
import numpy as np

# Load your dataset
my_df = pd.read_excel("../datasets/processed_data/university_final_dataset.xlsx")

# Load reference dataset
ref_df = pd.read_excel("../datasets/processed_data/reference_cleaned_dataset.xlsx")

print("My Dataset Shape :", my_df.shape)
print("Reference Dataset Shape :", ref_df.shape)

My Dataset Shape : (668, 95)
Reference Dataset Shape : (14161, 48)


In [5]:
print("My Dataset Columns:\n")
print(my_df.columns.tolist())

print("\n\nReference Dataset Columns:\n")
print(ref_df.columns.tolist())

My Dataset Columns:

['university_id', 'world_rank', '2023 RANK', 'university_name', 'country_code_iso', 'country', 'SIZE', 'FOCUS', 'RES.', 'AGE', 'STATUS', 'academic_reputation_score', 'Academic Reputation Rank', 'employer_reputation_score', 'Employer Reputation Rank', 'Faculty Student Score', 'Faculty Student Rank', 'citations_per_faculty', 'Citations per Faculty Rank', 'International Faculty Score', 'International Faculty Rank', 'International Students Score', 'International Students Rank', 'International Research Network Score', 'International Research Network Rank', 'Employment Outcomes Score', 'Employment Outcomes Rank', 'Sustainability Score', 'Sustainability Rank', 'overall_score', 'student_population', 'students_to_staff_ratio', 'international_students_count', 'gender_ratio', 'teaching_score', 'research_environment_score', 'research_quality_score', 'industry_impact_score', 'international_outlook_score', 'Quality of Education', 'Alumni Employment', 'Quality of Faculty', 'Resea

In [8]:
# Standardize university names

my_df["university_name"] = (
    my_df["university_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

ref_df["university_name"] = (
    ref_df["university_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

In [9]:
common_universities = set(my_df["university_name"]).intersection(
    set(ref_df["university_name"])
)

print("My Universities :", len(my_df))
print("Reference Universities :", len(ref_df))
print("Matching Universities :", len(common_universities))

My Universities : 668
Reference Universities : 14161
Matching Universities : 665


In [10]:
print(ref_df["year"].value_counts().sort_index())
print(ref_df["year"].unique())

year
2020    1925
2021    2519
2022    2894
2023    3226
2024    3597
Name: count, dtype: int64
[2020 2021 2022 2023 2024]


In [11]:
# Keep only 2024 data
ref_2024 = ref_df[ref_df["year"] == 2024].copy()

print("Rows after filtering:", len(ref_2024))
print("Unique Universities:", ref_2024["university_name"].nunique())

Rows after filtering: 3597
Unique Universities: 3548


In [13]:
# Check duplicate universities in 2024 data
duplicates = ref_2024["university_name"].duplicated().sum()

print("Duplicate Universities:", duplicates)

Duplicate Universities: 49


In [14]:
# Keep only the first occurrence
ref_2024 = ref_2024.drop_duplicates(subset="university_name")

print("Rows after removing duplicates:", len(ref_2024))
print("Unique Universities:", ref_2024["university_name"].nunique())

Rows after removing duplicates: 3548
Unique Universities: 3548


In [15]:
column_mapping = {
    "qs_rank": "QS_Rank",
    "the_rank": "THE_Rank",
    "qs_overall_score": "overall_score",
    "academic_reputation_score": "academic_reputation_score",
    "citations_per_faculty": "citations_per_faculty",
    "the_teaching": "teaching_score",
    "the_research_environment": "research_environment_score",
    "the_research_quality": "research_quality_score",
    "the_industry": "industry_impact_score",
    "the_international_outlook": "international_outlook_score",
    "students_per_staff": "students_to_staff_ratio",
    "faculty_student_ratio": "faculty_to_student_ratio",
    "international_students_pct": "international_student_ratio",
    "publications_count": "publications_count",
    "citations_count": "citations_count",
    "h_index": "h_index",
    "research_productivity_index": "research_productivity_index",
    "country_avg_rank": "country_avg_rank",
    "country_avg_overall_score": "country_avg_overall_score",
    "country_avg_academic_reputation": "country_avg_academic_reputation",
    "country_avg_citations": "country_avg_citations",
    "country_avg_intl_ratio": "country_avg_international_ratio",
    "female_pct": "female_percentage",
    "global_ranking_score": "global_ranking_score",
    "research_impact_score": "research_impact_score",
    "international_student_percentage": "international_student_percentage",
    "region": "region",
    "country": "country"
}

In [16]:
merged_df = my_df.merge(
    ref_2024[list(column_mapping.keys()) + ["university_name"]],
    on="university_name",
    how="left",
    suffixes=("", "_ref")
)

print("Merged Shape:", merged_df.shape)

Merged Shape: (668, 123)


In [17]:
for ref_col, my_col in column_mapping.items():
    print(f"{my_col} <-- {ref_col}")
    print("Available values:", merged_df[f"{ref_col}"].notna().sum())
    print("-"*40)

QS_Rank <-- qs_rank
Available values: 665
----------------------------------------
THE_Rank <-- the_rank
Available values: 665
----------------------------------------
overall_score <-- qs_overall_score
Available values: 665
----------------------------------------
academic_reputation_score <-- academic_reputation_score
Available values: 668
----------------------------------------
citations_per_faculty <-- citations_per_faculty
Available values: 668
----------------------------------------
teaching_score <-- the_teaching
Available values: 665
----------------------------------------
research_environment_score <-- the_research_environment
Available values: 665
----------------------------------------
research_quality_score <-- the_research_quality
Available values: 665
----------------------------------------
industry_impact_score <-- the_industry
Available values: 665
----------------------------------------
international_outlook_score <-- the_international_outlook
Available values: 6

In [21]:
# Create a copy of merged dataset
final_df = merged_df.copy()

replacement_report = {}

for ref_col, my_col in column_mapping.items():

    # After merge, reference columns with the same name get "_ref"
    source_col = ref_col

    if ref_col == my_col:
        source_col = ref_col + "_ref"

    if source_col in final_df.columns and my_col in final_df.columns:

        before = ((final_df[my_col] == -1) | (final_df[my_col].isna())).sum()

        final_df.loc[
            (final_df[my_col] == -1) | (final_df[my_col].isna()),
            my_col
        ] = final_df.loc[
            (final_df[my_col] == -1) | (final_df[my_col].isna()),
            source_col
        ]

        after = ((final_df[my_col] == -1) | (final_df[my_col].isna())).sum()

        replacement_report[my_col] = before - after

print("\nReplacement Summary")
print("-" * 50)

for col, count in replacement_report.items():
    print(f"{col:<40} {count}")
print("-"*40)

for col, count in replacement_report.items():
    print(f"{col:<40} {count}")


Replacement Summary
--------------------------------------------------
QS_Rank                                  653
THE_Rank                                 653
overall_score                            0
academic_reputation_score                0
citations_per_faculty                    0
teaching_score                           0
research_environment_score               0
research_quality_score                   0
industry_impact_score                    0
international_outlook_score              0
students_to_staff_ratio                  0
faculty_to_student_ratio                 653
international_student_ratio              653
publications_count                       0
citations_count                          0
h_index                                  4
research_productivity_index              0
country_avg_rank                         0
country_avg_overall_score                0
country_avg_academic_reputation          0
country_avg_citations                    0
country_avg_inter

C:\Users\yadav\AppData\Local\Temp\ipykernel_18196\3631444298.py:18: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[133.   21.   23.   28.   28.   30.   33.  152.   37.   72.   41.   42.
  43.  131.   46.  125.   94.   50.   51.   52.   53.   55.   57.   58.
  59.   60.   61.   63.   64.   69.   70.   71.   72.   94.   75.   92.
  78.   79.   80.   92.   84.  124.  308.5  90.  119.5  94.   92.   97.
  92.  102.  105.  106.  138.  109.  110.  111.  112.  113.  114.  115.
 116.  117.  118.  121.  122.  123.  125.  119.5 125.  119.5 128.  129.
 133.  354.  118.  316.   94.  136.  131.  139.  140.  131.  138.   92.
  94.  148.  125.   92.  188.  162.   79.  136.  167.  168.   94.   92.
 354.  195.  224.  294.5 136.   92.  182.  183.   72.  187.  152.  194.
 196.  166.  136.  109.  152.  206.  207.  208.  210.  211.  212.  214.
 215.  216.  217.  218.  119.5 224.  225.  228.  277.  232.  133.  189.5
 235.   94

In [22]:
negative_columns = (
    final_df.select_dtypes(include="number") < 0
).sum()

print(negative_columns[negative_columns > 0])

Latitude                      23
Longitude                    510
total_students               656
undergraduate_students       656
postgraduate_students        656
tuition_fee_usd              656
acceptance_rate              656
graduation_rate              656
Employability_Rank           656
qs_subject_rank              656
ARWU_Rank                    656
faculty_student_ratio_kpi    656
dtype: int64


In [23]:
print(final_df[["international_student_percentage", "international_student_percentage_ref"]].head(20))

    international_student_percentage  international_student_percentage_ref
0                               38.0                                  38.0
1                               43.0                                  42.0
2                               25.0                                  25.0
3                               23.0                                  23.0
4                               60.0                                  61.0
5                               53.0                                  61.0
6                               24.0                                  22.0
7                               24.0                                  24.0
8                               22.0                                  21.0
9                               15.0                                  14.0
10                              23.0                                  23.0
11                              51.0                                  51.0
12                       

In [25]:
print(final_df["faculty_count"].dtype)
print(final_df["student_population"].dtype)

object
int64


In [26]:
print(final_df["faculty_count"].head(10))
print(final_df["student_population"].head(10))

0    5,800
1    6,708
2    4,646
3    4,725
4    8,000
5    7,195
6    5,154
7    2,843
8    5,391
9    5,302
Name: faculty_count, dtype: object
0    20565
1    21750
2    20050
3    14517
4    20275
5    41110
6    21312
7    23620
8    11924
9    33064
Name: student_population, dtype: int64


In [27]:
import pandas as pd

# Remove commas if present and convert to numeric
final_df["faculty_count"] = pd.to_numeric(
    final_df["faculty_count"].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
)

final_df["student_population"] = pd.to_numeric(
    final_df["student_population"].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
)

In [28]:
final_df["faculty_student_ratio_kpi"] = (
    final_df["faculty_count"] / final_df["student_population"]
).round(4)

In [30]:
final_df.to_excel(
    "../datasets/processed_data/university_final_dataset_enriched.xlsx",
    index=False
)

print("Enriched dataset saved successfully!")

Enriched dataset saved successfully!


In [32]:
ref2 = pd.read_excel("../datasets/raw_data/ref_data.xlsx")

print(ref2.shape)
print(ref2.columns.tolist())

(22125, 44)
['university_id', 'year', 'university_name', 'world_rank', 'national_rank', 'overall_score', 'country', 'region', 'city', 'university_type', 'academic_reputation_score', 'employer_reputation_score', 'citations_score', 'publications_count', 'citations_count', 'citations_per_faculty', 'h_index', 'research_output_score', 'research_productivity_index', 'subject_field', 'total_students', 'international_students_count', 'international_student_ratio', 'faculty_count', 'faculty_to_student_ratio', 'gender_ratio', 'female_percentage', 'male_percentage', 'degree_level', 'undergraduate_count', 'postgraduate_count', 'country_avg_rank', 'universities_ranked_count', 'best_university_rank', 'country_avg_overall_score', 'country_avg_academic_reputation', 'country_avg_citations', 'country_avg_international_ratio', 'global_ranking_score', 'research_impact_score', 'faculty_student_ratio_kpi', 'international_student_percentage_kpi', 'academic_reputation_kpi', 'research_productivity_kpi']


In [35]:
ref2["university_name"] = (
    ref2["university_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

In [34]:
common = set(final_df["university_name"]).intersection(
    set(ref2["university_name"])
)

print("Matching:", len(common))

Matching: 625


In [36]:
remaining_columns = [
    "total_students",
    "undergraduate_students",
    "postgraduate_students",
    "faculty_to_student_ratio",
    "faculty_student_ratio_kpi",
    "degree_level",
    "international_students_count",
    "international_student_ratio",
    "female_percentage",
    "male_percentage",
    "gender_ratio"
]

In [37]:
mapping2 = {
    "undergraduate_count":"undergraduate_students",
    "postgraduate_count":"postgraduate_students",
    "total_students":"total_students",
    "faculty_to_student_ratio":"faculty_to_student_ratio",
    "faculty_student_ratio_kpi":"faculty_student_ratio_kpi",
    "degree_level":"degree_level",
    "international_students_count":"international_students_count",
    "international_student_ratio":"international_student_ratio",
    "female_percentage":"female_percentage",
    "male_percentage":"male_percentage",
    "gender_ratio":"gender_ratio"
}

In [38]:
merged_df2 = final_df.merge(
    ref2[list(mapping2.keys()) + ["university_name"]],
    on="university_name",
    how="left",
    suffixes=("", "_ref2")
)

In [39]:
final_df2 = merged_df2.copy()

replacement_report = {}

for ref_col, my_col in mapping2.items():

    source_col = ref_col

    # If both datasets have the same column name,
    # pandas adds "_ref2" to the reference column.
    if ref_col == my_col:
        source_col = ref_col + "_ref2"

    if source_col in final_df2.columns and my_col in final_df2.columns:

        before = (final_df2[my_col] == -1).sum()

        mask = (
            (final_df2[my_col] == -1)
            &
            (final_df2[source_col] != -1)
            &
            (final_df2[source_col].notna())
        )

        final_df2.loc[mask, my_col] = final_df2.loc[mask, source_col]

        after = (final_df2[my_col] == -1).sum()

        replacement_report[my_col] = before - after

C:\Users\yadav\AppData\Local\Temp\ipykernel_18196\783311525.py:26: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[42678. 57414. 44553. ... 11089. 11428. 11560.]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  final_df2.loc[mask, my_col] = final_df2.loc[mask, source_col]


In [40]:
print("Replacement Report\n")

for col, count in replacement_report.items():
    print(f"{col}: {count} values filled")

Replacement Report

undergraduate_students: 6089 values filled
postgraduate_students: 6089 values filled
total_students: 6089 values filled
faculty_to_student_ratio: 0 values filled
faculty_student_ratio_kpi: 0 values filled
degree_level: 0 values filled
international_students_count: 0 values filled
international_student_ratio: 0 values filled
female_percentage: 0 values filled
male_percentage: 0 values filled
gender_ratio: 0 values filled


In [41]:
ref_cols = [col for col in final_df2.columns if col.endswith("_ref2")]

final_df2.drop(columns=ref_cols, inplace=True)

print("Temporary reference columns removed.")

Temporary reference columns removed.


In [42]:
negative_counts = (final_df2 == -1).sum()

negative_counts = negative_counts[negative_counts > 0]

print(negative_counts.sort_values())

total_students              43
undergraduate_students      43
postgraduate_students       43
tuition_fee_usd           6132
acceptance_rate           6132
graduation_rate           6132
Employability_Rank        6132
qs_subject_rank           6132
ARWU_Rank                 6132
dtype: int64


In [43]:
final_df2.to_excel(
    "../datasets/final_dataset/university_final_dataset.xlsx",
    index=False
)

print("Final dataset saved successfully!")

Final dataset saved successfully!


In [44]:
# Remove every temporary reference column
cols_to_remove = [
    col for col in final_df2.columns
    if col.endswith("_ref")
    or col.endswith("_ref2")
]

final_df2.drop(columns=cols_to_remove, inplace=True)

print("Removed", len(cols_to_remove), "temporary columns")

Removed 15 temporary columns


In [45]:
duplicates = [
    "qs_rank",
    "the_rank",
    "qs_overall_score",
    "undergraduate_count",
    "postgraduate_count"
]

duplicates = [c for c in duplicates if c in final_df2.columns]

final_df2.drop(columns=duplicates, inplace=True)

print("Duplicate helper columns removed.")

Duplicate helper columns removed.


In [46]:
print(final_df2.shape)

(6264, 105)


In [52]:
final_df2.to_excel(
    "../datasets/final_dataset/university_final_dataset.xlsx",
    index=False
)

print("Final dataset saved successfully!")

Final dataset saved successfully!


In [53]:
ref_cols = [col for col in final_df2.columns if "_ref" in col]

print(ref_cols)
print("Count:", len(ref_cols))

[]
Count: 0


In [54]:
duplicate_columns = final_df2.columns[final_df2.columns.duplicated()]

print(duplicate_columns)

Index([], dtype='object')


In [55]:
print(final_df2.info())
print(final_df2.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6264 entries, 0 to 6263
Columns: 105 entries, university_id to female_pct
dtypes: float64(54), int64(23), object(28)
memory usage: 5.0+ MB
None
   university_id world_rank 2023 RANK          university_name  \
0              1          2         2  university of cambridge   
1              1          2         2  university of cambridge   
2              1          2         2  university of cambridge   
3              1          2         2  university of cambridge   
4              1          2         2  university of cambridge   

  country_code_iso         country SIZE FOCUS RES.  AGE  ... the_teaching  \
0               UK  United Kingdom    L    FC   VH    5  ...         95.8   
1               UK  United Kingdom    L    FC   VH    5  ...         95.8   
2               UK  United Kingdom    L    FC   VH    5  ...         95.8   
3               UK  United Kingdom    L    FC   VH    5  ...         95.8   
4               UK  Unit

In [56]:
required_kpis = [
    "global_ranking_score",
    "research_impact_score",
    "faculty_to_student_ratio",
    "international_student_percentage",
    "academic_reputation_score",
    "research_productivity_index"
]

for col in required_kpis:
    print(f"{col}: {'✅ Present' if col in final_df2.columns else '❌ Missing'}")

global_ranking_score: ✅ Present
research_impact_score: ✅ Present
faculty_to_student_ratio: ✅ Present
international_student_percentage: ✅ Present
academic_reputation_score: ✅ Present
research_productivity_index: ✅ Present


In [57]:
final_df2[
    [
        "global_ranking_score",
        "research_impact_score",
        "faculty_to_student_ratio",
        "international_student_percentage",
        "academic_reputation_score",
        "research_productivity_index"
    ]
].describe()

,global_ranking_score,research_impact_score,faculty_to_student_ratio,international_student_percentage,academic_reputation_score,research_productivity_index
count,6261.000000,6264.000000,6261.000000,6261.000000,6264.000000,6264.000000
mean,424.662303,293.687292,17.894194,14.847309,25.344923,52.557886
std,354.801533,137.382881,8.980044,12.775376,24.857247,19.726029
min,26.529730,32.166667,3.900000,0.000000,1.600000,11.933333
25%,57.821622,225.533333,12.300000,5.000000,7.800000,37.366667
50%,470.000000,273.661111,16.100000,11.000000,15.500000,53.633333
75%,758.000000,370.155556,20.800000,23.000000,32.400000,66.933333
max,998.000000,681.922222,114.200000,92.000000,100.000000,99.233333
